In [2]:
# Merge GT and Pred CSVs, compute Passable and Correct, and save results
import pandas as pd
import numpy as np
import os

# --- Configure file paths (update as needed) ---
base_dir = r"C:\Users\kyohe\Aerial_Photo_Segmenter\20260105Data\Result_QGIS"
GT_csv = "GT_wajima_overall.csv"  # Ground Truth CSV (File A)
Pred_csv = "Pred_wajima_overall.csv"  # Prediction CSV (File B)
output_csv = "merged_wajima_overall.csv"

GT_csv = os.path.join(base_dir, GT_csv)
Pred_csv =  os.path.join(base_dir, Pred_csv)
output_csv = os.path.join(base_dir, output_csv)

# --- Required columns ---
required_cols = ['filename', 'angle_deg_clockwise', 'width_m', 'original_road_width_m', 'remaining_road_width_m']

# 1) Read CSVs
dfA = pd.read_csv(GT_csv)
dfB = pd.read_csv(Pred_csv)

# 2) Basic validation: ensure required columns exist
missing_GT = [c for c in required_cols if c not in dfA.columns]
missing_Pred = [c for c in required_cols if c not in dfB.columns]
if missing_GT:
    raise KeyError(f"File A is missing required columns: {missing_GT}")
if missing_Pred:
    raise KeyError(f"File B is missing required columns: {missing_Pred}")

# 3) Select and rename columns with prefixes (keep `filename`)
dfA_sel = dfA[required_cols].copy()
dfB_sel = dfB[required_cols].copy()

dfA_sel = dfA_sel.rename(columns={c: f"GT_{c}" for c in dfA_sel.columns if c != 'filename'})
dfB_sel = dfB_sel.rename(columns={c: f"Pred_{c}" for c in dfB_sel.columns if c != 'filename'})

# 4) Inner join on `filename`
df = pd.merge(dfA_sel, dfB_sel, on='filename', how='inner')

# 5) Compute Passable columns (preserve NaN): >4.0 -> True, <=4.0 -> False, NaN -> NaN
df['GT_Passable'] = np.where(df['GT_remaining_road_width_m'].isna(), np.nan, df['GT_remaining_road_width_m'] > 4.0)
df['Pred_Passable'] = np.where(df['Pred_remaining_road_width_m'].isna(), np.nan, df['Pred_remaining_road_width_m'] > 4.0)

# 6) Compute Correct column: NaN if either is NaN, else True if equal, False otherwise
df['Correct'] = np.where(df['GT_Passable'].isna() | df['Pred_Passable'].isna(), np.nan, df['GT_Passable'] == df['Pred_Passable'])

# 7) Save to CSV (without index) and keep DataFrame variable for later cells
df.to_csv(output_csv, index=False)

print(f"Saved merged results to: {output_csv} (rows: {len(df)})")

# Keep DataFrame variable for reuse in later cells
merged_gt_pred_df = df

# Show a quick preview
merged_gt_pred_df.head()

Saved merged results to: C:\Users\kyohe\Aerial_Photo_Segmenter\20260105Data\Result_QGIS\merged_wajima_overall.csv (rows: 28)


,filename,GT_angle_deg_clockwise,GT_width_m,GT_original_road_width_m,GT_remaining_road_width_m,Pred_angle_deg_clockwise,Pred_width_m,Pred_original_road_width_m,Pred_remaining_road_width_m,GT_Passable,Pred_Passable,Correct
0,b10_HouseCollapse_110_clipped_f1_bbox_f0.gpkg,185.325741,3.542571,11.862516,8.319945,2.624220,1.227862,11.939776,10.711913,1.0,1.0,1.0
1,b10_HouseCollapse_127_clipped_f1_bbox_f0.gpkg,87.482082,1.433516,4.724743,3.291227,91.637475,1.639596,4.968200,3.328604,0.0,0.0,1.0
2,b10_HouseCollapse_160_clipped_f1_bbox_f0.gpkg,174.233233,11.703779,12.286887,0.583108,235.565092,2.502957,5.702369,3.199412,0.0,0.0,1.0
3,b10_HouseCollapse_167_clipped_f1_bbox_f0.gpkg,29.390981,3.590224,4.604332,1.014107,173.134559,2.860740,3.582035,0.721295,0.0,0.0,1.0
4,b10_HouseCollapse_194_clipped_f1_bbox_f0.gpkg,258.056219,3.352781,4.587818,1.235037,168.067671,6.768256,NaN,NaN,0.0,NaN,NaN
